# Лабораторная работа 4. Поиск подстроки

**Опора:** лекция 7.

Реализуются **три алгоритма поиска**: наивный, Кнута–Морриса–Пратта и
Рабина–Карпа. Задача у всех трёх одна.

- **Вход:** текст и образец.
- **Выход:** список позиций начала всех вхождений образца в текст, в порядке
  возрастания, включая перекрывающиеся вхождения. Если вхождений нет — пустой
  список.

```text
Вход:  текст = "ababa", образец = "aba"      Выход: [0, 2]
Вход:  текст = "abcdef", образец = "xyz"     Выход: []
```

Пустой образец считается входящим в каждую позицию, включая позицию за концом
текста: для текста длины `n` ответ — `[0, 1, …, n]`.

Тексты, образцы, измерение времени и графики уже готовы и
лежат в пакете `labkit` рядом с ноутбуком.

In [ ]:
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "labkit").is_dir():
        sys.path.insert(0, str(candidate))
        break

import labkit as lk

# Полные размеры из условия работы: раскомментировать следующую строку.
# lk.use_full_sizes()

print("Режим:", lk.CONFIG.name, "| длины текстов:", lk.text_sizes())

## Задача 1. Наивный поиск

Образец прикладывается к каждой позиции текста и сравнивается посимвольно.

In [ ]:
def naive_search(text, pattern):
    """Список позиций всех вхождений pattern в text, включая перекрывающиеся."""
    # TODO: реализовать наивный поиск
    raise NotImplementedError("naive_search")

In [ ]:
lk.check_search(naive_search, "Наивный поиск")

## Задача 2. Алгоритм Кнута–Морриса–Пратта

Отдельно вычисляется префикс-функция образца: для каждого префикса — длина его
наибольшего собственного суффикса, который одновременно является его префиксом.
При несовпадении поиск использует префикс-функцию и не возвращается назад по
тексту.

```text
Вход префикс-функции:  "ababa"    Выход: [0, 0, 1, 2, 3]
```

In [ ]:
def prefix_function(pattern):
    """Для каждого префикса — длина наибольшего собственного суффикса,
    который одновременно является его префиксом."""
    # TODO: реализовать
    raise NotImplementedError("prefix_function")


def kmp_search(text, pattern):
    """Список позиций всех вхождений pattern в text."""
    # TODO: реализовать поиск Кнута–Морриса–Пратта
    raise NotImplementedError("kmp_search")

In [ ]:
lk.check_prefix_function(prefix_function)
lk.check_search(kmp_search, "Кнут–Моррис–Пратт")

## Задача 3. Алгоритм Рабина–Карпа

Сравниваются хеши образца и окна текста той же длины. Хеш здесь тот же
полиномиальный, что и у строковых ключей хеш-таблицы, — схема Горнера с
основанием `X` и модулем `q`:

```text
h = 0
для каждого символа c: h = (h * X + ord(c)) mod q
```

Отличие в том, что для каждого следующего окна хеш не вычисляется заново, а
пересчитывается за `O(1)` при сдвиге на один символ: из хеша вычитается вклад
ушедшего символа, остаток умножается на основание, прибавляется код пришедшего.
Такой хеш называется скользящим. Основание берётся не меньше размера алфавита,
модуль — большое простое число.

- **Требование:** при совпадении хешей обязательна проверка реального совпадения
  строк, поскольку возможны коллизии.

In [ ]:
def rabin_karp_search(text, pattern):
    """Список позиций всех вхождений pattern в text."""
    # TODO: реализовать поиск Рабина–Карпа
    raise NotImplementedError("rabin_karp_search")

In [ ]:
lk.check_search(rabin_karp_search, "Рабин–Карп")

## Набор реализованных алгоритмов

Дальше всё считается по тем реализациям, которые действительно написаны. Все три
должны давать одинаковый ответ на одних и тех же данных.

In [ ]:
SEARCHES = lk.implemented_searches({
    "наивный": naive_search,
    "Кнут–Моррис–Пратт": kmp_search,
    "Рабин–Карп": rabin_karp_search,
})

print("В экспериментах участвуют:", ", ".join(SEARCHES))
lk.check_same_results(SEARCHES, lk.random_text(5_000), ["abc", "abcd", "zzz", "a"])

## Эксперимент 1. Размер входа

Слева растёт длина текста при образце длины 20, взятом из текста; справа
длина текста фиксирована, а растёт длина образца.

In [ ]:
by_size = lk.size_experiment(SEARCHES)
lk.plot_search_experiment(by_size, "Размер входа · время поиска");

## Эксперимент 2. Масштабирование алгоритмов

Длины текста `n` и образца `m` удваиваются одновременно, причём во всех точках
`m = n / 10`. Алгоритмы проверяются на случайном тексте, на входе, где
несовпадение находится примерно после ¾ образца, и на тексте с большим числом
вхождений.

Каждый столбец соответствует алгоритму. Сверху показано медианное время поиска,
снизу — то же время, делённое на суммарную длину входа `n + m`. Полоса вокруг
линии показывает интервал от первого до третьего квартиля.

In [ ]:
scaling = lk.scaling_experiment(SEARCHES)
lk.plot_scaling(scaling);

## Самостоятельные выводы

1. Почему увеличение длины текста и длины образца по-разному влияет на время
   поиска? Учтите число проверяемых окон `n − m + 1` и предварительную обработку
   образца.
2. Почему на случайном тексте наивный поиск может быть сопоставим с КМП и
   Рабином–Карпом, а на входе с длинным несовпадением начинает заметно
   проигрывать?
3. Как префикс-функция помогает КМП не возвращаться назад по тексту и находить
   перекрывающиеся вхождения?
4. Как скользящий хеш позволяет Рабину–Карпу обрабатывать следующее окно за
   `O(1)` и почему при совпадении хешей всё равно необходимо сравнивать строки?
5. Какие результаты графиков объясняются асимптотикой алгоритмов, а какие —
   устройством конкретных входов и постоянными затратами реализации на
   Python?